# Step 2: check the rebuild against the official flyvis

This notebook installs the official flyvis package, downloads its pretrained networks, runs networks 000 and 003 on the same test stimuli as the rebuild, and compares the numbers row by row.

**To run:** Runtime → Run all. No GPU is needed. If Colab asks you to restart the session after the install, restart and choose Run all again.


In [ ]:
# 1. Install flyvis 1.2.0 (it supports Python 3.9 to 3.12)
import sys
print('Colab Python', sys.version.split()[0])
if sys.version_info < (3, 13):
    !pip install -q flyvis==1.2.0
else:
    # Colab's Python is too new for flyvis, so build a separate Python 3.12 environment
    !pip install -q uv
    !uv venv /content/py312 --python 3.12 -q
    !uv pip install -q --python /content/py312/bin/python flyvis==1.2.0
print('install finished')


In [ ]:
# 2. Download the 50 pretrained networks (skips the large clustering files)
import os, sys
os.environ["FLYVIS_ROOT_DIR"] = "/content/flyvis_data"
if os.path.exists("/content/py312/bin/python"):
    PY, FLYVIS = "/content/py312/bin/python", "/content/py312/bin/flyvis"
else:
    PY, FLYVIS = sys.executable, "flyvis"
!{FLYVIS} download-pretrained --skip_large_files


In [ ]:
%%writefile official_check.py
"""Step 2: run the official flyvis on the rebuild's test stimuli and compare the numbers.

Run with the Python that has flyvis 1.2.0 installed, after `flyvis download-pretrained --skip_large_files`.
"""
import hashlib
import json

import numpy as np
import torch

import flyvis
from flyvis.datasets.rendering import BoxEye

EXPECTED = json.loads(r'''{"000": {"grey_drift": 0.000438690185546875, "grey_range": 6.096230506896973, "R1_corr": 0.9393337667330502, "ON_T4": 2.8263707160949707, "ON_T5": 0.5910622477531433, "OFF_T4": 0.33386409282684326, "OFF_T5": 0.9360684752464294, "settle_diff": 0.00035268068313598633, "fingerprint_types": ["R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8", "L1", "L2", "L3", "L4", "L5", "Lawf1", "Lawf2", "Am", "C2", "C3", "CT1(Lo1)", "CT1(M10)", "Mi1", "Mi2", "Mi3", "Mi4", "Mi9", "Mi10", "Mi11", "Mi12", "Mi13", "Mi14", "Mi15", "T1", "T2", "T2a", "T3", "T4a", "T4b", "T4c", "T4d", "T5a", "T5b", "T5c", "T5d", "Tm1", "Tm2", "Tm3", "Tm4", "Tm5Y", "Tm5a", "Tm5b", "Tm5c", "Tm9", "Tm16", "Tm20", "Tm28", "Tm30", "TmY3", "TmY4", "TmY5a", "TmY9", "TmY10", "TmY13", "TmY14", "TmY15", "TmY18"], "fingerprint_on_edge_final": [1.764192, 1.637918, 1.37741, 1.861889, 2.167302, 2.31685, 0.431134, 1.777668, -2.630837, -2.631357, 0.036589, 1.092171, 0.091642, -0.466214, 0.440992, -3.742183, 0.251095, 0.244174, 0.219228, 1.095099, 0.835344, 1.229897, 0.501203, 2.545579, -0.022156, 1.252693, 0.694535, 0.683411, 0.680029, 0.297928, 0.250096, -0.342306, 5.553638, 0.418614, 0.604761, 0.033718, -0.027842, -0.064043, -0.182472, 0.053074, 0.007382, -0.004668, -0.037419, 0.251461, -0.298782, 1.606106, -0.067044, 0.240045, -0.193523, 0.248553, 1.496649, -0.06314, -0.228913, 0.468626, -0.057383, 0.356926, -0.802206, -0.056016, 0.736883, 1.441786, 0.231577, 0.392323, -1.055452, 0.602722, 1.74593]}, "003": {"grey_drift": 0.08599856495857239, "grey_range": 11.110447883605957, "R1_corr": 0.9442645589452991, "ON_T4": 8.655974388122559, "ON_T5": 3.123133659362793, "OFF_T4": 0.004754841327667236, "OFF_T5": 3.241910696029663, "settle_diff": 0.017287850379943848, "fingerprint_types": ["R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8", "L1", "L2", "L3", "L4", "L5", "Lawf1", "Lawf2", "Am", "C2", "C3", "CT1(Lo1)", "CT1(M10)", "Mi1", "Mi2", "Mi3", "Mi4", "Mi9", "Mi10", "Mi11", "Mi12", "Mi13", "Mi14", "Mi15", "T1", "T2", "T2a", "T3", "T4a", "T4b", "T4c", "T4d", "T5a", "T5b", "T5c", "T5d", "Tm1", "Tm2", "Tm3", "Tm4", "Tm5Y", "Tm5a", "Tm5b", "Tm5c", "Tm9", "Tm16", "Tm20", "Tm28", "Tm30", "TmY3", "TmY4", "TmY5a", "TmY9", "TmY10", "TmY13", "TmY14", "TmY15", "TmY18"], "fingerprint_on_edge_final": [1.080044, 1.027975, 1.051133, 0.34919, 0.436274, 0.394563, 1.502047, 1.00481, 1.314585, 3.042682, 0.492363, 0.129002, 3.580718, 1.172007, 4.92305, 1.768609, 15.100884, -0.974404, 3.654461, 2.645056, 5.968703, 7.196629, 0.48657, 8.481285, -2.383136, 2.18909, 0.860571, 1.201397, 0.315834, 7.067865, 1.987087, 1.280035, 26.322325, 8.284203, -0.562765, -0.576058, -0.613044, 0.207526, 0.050955, 1.668583, 8.638713, 0.101652, -0.021283, 1.369197, 5.719755, 9.623268, -0.533849, 0.901075, -1.914759, -0.222195, -2.097502, 2.294003, 0.620392, 0.183457, 0.573927, 0.53956, -13.173175, -5.305334, -0.439456, 1.332641, -7.945238, -1.324721, 0.115392, 5.211799, -2.499267]}}''')
CHECKPOINT_SHA256 = {
    "000": "d8a57a022aa18ca338599be713af9db29b89cde4b00098ffe42b78f0d583e46a",
    "003": "b6764982671b5f0051e682360adc52f8ed6493e478fb9c0edbf66f1d256e1501",
}
DT = 1 / 100
T4, T5 = ["T4a", "T4b", "T4c", "T4d"], ["T5a", "T5b", "T5c", "T5d"]
eye = BoxEye(extent=15, kernel_size=13)


def edge_clip(on=True, n=60, size=391):
    clip = np.full((n, size, size), 0.0 if on else 1.0, np.float32)
    for i in range(n):
        clip[i, :, : int(size * i / n)] = 1.0 if on else 0.0
    return clip


def to_steps(hex_frames, fps):
    return hex_frames[(np.arange(int(len(hex_frames) / fps / DT)) * DT * fps).astype(int)]


def render(frames):
    """(n, 391, 391) frames -> (n, 721) eye input, with flyvis's own BoxEye."""
    return eye(frames[None]).cpu().numpy()[0, :, 0, :]


def run(net, movie):
    """(T, 721) eye input -> (T, 45669) activity, starting from flyvis's fade-in state."""
    x = torch.as_tensor(movie, dtype=torch.float32, device=flyvis.device)[None, :, None, :]
    with torch.no_grad():
        state = net.fade_in_state(1.0, DT, x[:, 0])
        return net.simulate(x, DT, initial_state=state).cpu().numpy()[0]


def close(a, b):
    return abs(a - b) <= max(1e-3, 1e-3 * abs(a))


print(f"flyvis {getattr(flyvis, '__version__', '?')}, torch {torch.__version__}, device {flyvis.device}\n")
on = to_steps(render(edge_clip(True)), 60)
off = to_steps(render(edge_clip(False)), 60)
rows, all_ok = [], True
for model in ["000", "003"]:
    exp = EXPECTED[model]
    path = flyvis.results_dir / "flow" / "0000" / model / "best_chkpt"
    sha = hashlib.sha256(path.read_bytes()).hexdigest()
    same_file = sha == CHECKPOINT_SHA256[model]
    all_ok &= same_file
    rows.append((model, "checkpoint file (SHA-256)", CHECKPOINT_SHA256[model][:10], sha[:10],
                 "", "MATCH" if same_file else "DIFFERENT"))

    net = flyvis.NetworkView(flyvis.results_dir / "flow" / "0000" / model).init_network()
    ct = net.connectome
    types, u, v = ct.nodes.type[:].astype(str), ct.nodes.u[:], ct.nodes.v[:]
    order = exp["fingerprint_types"]
    central = np.array([np.flatnonzero((types == t) & (u == 0) & (v == 0))[0] for t in order])
    col = {t: i for i, t in enumerate(order)}
    peak = lambda tr, cs: float(np.max(tr[:, [col[c] for c in cs]] - tr[0, [col[c] for c in cs]]))

    grey = run(net, np.full((100, 721), 0.5, np.float32))[:, central]
    rng = np.random.default_rng(int(model))
    flick = np.repeat(rng.uniform(0, 1, (20, 721)), 5, axis=0).astype(np.float32)
    r1 = run(net, flick)[:, np.flatnonzero(types == "R1")]
    a_on, a_off = run(net, on)[:, central], run(net, off)[:, central]
    after = run(net, np.concatenate([on, np.full((150, 721), 0.5, np.float32)]))[:, central]
    ref = run(net, np.full((250, 721), 0.5, np.float32))[:, central]
    got = {
        "grey_drift": float(np.max(grey[50:].max(0) - grey[50:].min(0))),
        "grey_range": float(grey.max() - grey.min()),
        "R1_corr": float(np.median([np.corrcoef(r1[:, c], flick[:, c])[0, 1] for c in range(721)])),
        "ON_T4": peak(a_on, T4), "ON_T5": peak(a_on, T5),
        "OFF_T4": peak(a_off, T4), "OFF_T5": peak(a_off, T5),
        "settle_diff": float(np.max(np.abs(after[-1] - ref[-1]))),
    }
    names = {"grey_drift": "grey: drift in last 0.5 s", "grey_range": "grey: activity range",
             "R1_corr": "R1 vs input (median corr)", "ON_T4": "ON edge: T4 peak", "ON_T5": "ON edge: T5 peak",
             "OFF_T4": "OFF edge: T4 peak", "OFF_T5": "OFF edge: T5 peak", "settle_diff": "settle: gap to grey"}
    for k, name in names.items():
        ok = close(exp[k], got[k]); all_ok &= ok
        rows.append((model, name, f"{exp[k]:.4f}", f"{got[k]:.4f}", f"{abs(exp[k] - got[k]):.1e}",
                     "match" if ok else "DIFFERENT"))
    fp = float(np.max(np.abs(np.array(exp["fingerprint_on_edge_final"]) - a_on[-1])))
    ok = fp <= 1e-3; all_ok &= ok
    rows.append((model, "all 65 cell types, end of ON edge", "", "", f"{fp:.1e}", "match" if ok else "DIFFERENT"))

w = [7, 36, 11, 11, 10, 9]
head = ("network", "measurement", "rebuild", "official", "diff", "result")
print("  ".join(h.ljust(n) for h, n in zip(head, w)))
for r in rows:
    print("  ".join(str(c).ljust(n) for c, n in zip(r, w)))
print("\n" + ("ALL MATCH: the rebuild reproduces the official flyvis model."
              if all_ok else "SOME ROWS DIFFER: paste this whole output back into the chat."))


In [ ]:
# 4. Run the comparison
import os, sys
os.environ["FLYVIS_ROOT_DIR"] = "/content/flyvis_data"
if os.path.exists("/content/py312/bin/python"):
    PY, FLYVIS = "/content/py312/bin/python", "/content/py312/bin/flyvis"
else:
    PY, FLYVIS = sys.executable, "flyvis"
!{PY} official_check.py


## Reading the result

- **Checksum rows say MATCH:** the networks you downloaded are byte for byte the ones the rebuild's numbers came from.
- **Measurement rows say match:** the official code gives the same numbers, within 0.001. Tiny differences (around 0.000001 to 0.0001) are normal rounding.
- **The last line says ALL MATCH:** you're done. Otherwise, paste the whole output back into the chat.
